# NLP Weekly Task 4: Bag of Words and TF-IDF

This notebook demonstrates two core text-vectorization techniques in NLP:

1. **Bag of Words (BoW)** — counts how many times each word appears in a document
2. **TF-IDF** — weighs words by importance (frequent in a document, but rare across the whole corpus)

We implement both using scikit-learn's `CountVectorizer` and `TfidfVectorizer`, and also walk through a small "from scratch" version so the underlying math is clear.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 1. Sample Corpus

In [2]:
corpus = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "Cats and dogs are great pets",
    "The dog chased the cat"
]

for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")

Doc 0: The cat sat on the mat
Doc 1: The dog sat on the log
Doc 2: Cats and dogs are great pets
Doc 3: The dog chased the cat


## 2. Bag of Words (`CountVectorizer`)

BoW represents each document as a vector of raw word counts, ignoring grammar and word order.

In [3]:
bow_vectorizer = CountVectorizer(lowercase=True, stop_words=None)
bow_matrix = bow_vectorizer.fit_transform(corpus)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)

print("Vocabulary size:", len(bow_vectorizer.vocabulary_))
bow_df

Vocabulary size: 14


,and,are,cat,cats,chased,dog,dogs,great,log,mat,on,pets,sat,the
Doc 0,0,0,1,0,0,0,0,0,0,1,1,0,1,2
Doc 1,0,0,0,0,0,1,0,0,1,0,1,0,1,2
Doc 2,1,1,0,1,0,0,1,1,0,0,0,1,0,0
Doc 3,0,0,1,0,1,1,0,0,0,0,0,0,0,2


## 3. TF-IDF (`TfidfVectorizer`)

TF-IDF = Term Frequency × Inverse Document Frequency. It downweights common words (like "the") that appear in most documents, and upweights words that are distinctive to a particular document.

In [4]:
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words=None)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
).round(3)

tfidf_df

,and,are,cat,cats,chased,dog,dogs,great,log,mat,on,pets,sat,the
Doc 0,0.000,0.000,0.372,0.000,0.000,0.000,0.000,0.000,0.000,0.472,0.372,0.000,0.372,0.602
Doc 1,0.000,0.000,0.000,0.000,0.000,0.372,0.000,0.000,0.472,0.000,0.372,0.000,0.372,0.602
Doc 2,0.408,0.408,0.000,0.408,0.000,0.000,0.408,0.408,0.000,0.000,0.000,0.408,0.000,0.000
Doc 3,0.000,0.000,0.401,0.000,0.508,0.401,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.649


In [5]:
# Top 3 highest-weighted words per document
for i in range(len(corpus)):
    row = tfidf_df.iloc[i]
    top_words = row.sort_values(ascending=False).head(3)
    print(f"Doc {i}: {list(zip(top_words.index, top_words.values))}")

Doc 0: [('the', np.float64(0.602)), ('mat', np.float64(0.472)), ('sat', np.float64(0.372))]
Doc 1: [('the', np.float64(0.602)), ('log', np.float64(0.472)), ('sat', np.float64(0.372))]
Doc 2: [('and', np.float64(0.408)), ('are', np.float64(0.408)), ('cats', np.float64(0.408))]
Doc 3: [('the', np.float64(0.649)), ('chased', np.float64(0.508)), ('dog', np.float64(0.401))]


## 4. Manual TF-IDF Calculation (for understanding the math)

$$TF(word, doc) = \frac{\text{count of word in doc}}{\text{total words in doc}}$$

$$IDF(word) = \ln\left(\frac{1 + N}{1 + df(word)}\right) + 1$$

$$TFIDF(word, doc) = TF(word, doc) \times IDF(word)$$

This is the smoothed formula scikit-learn uses internally (before L2 row normalization).

In [6]:
def compute_tf(word, document):
    words = document.lower().split()
    return words.count(word) / len(words)


def compute_idf(word, all_documents):
    n_docs_containing_word = sum(
        1 for doc in all_documents if word in doc.lower().split()
    )
    # +1 smoothing to avoid division by zero (same idea sklearn uses)
    return np.log((1 + len(all_documents)) / (1 + n_docs_containing_word)) + 1


word_to_check = "cat"
print(f"Manually computing TF-IDF for the word: '{word_to_check}'\n")
for i, doc in enumerate(corpus):
    tf = compute_tf(word_to_check, doc)
    idf = compute_idf(word_to_check, corpus)
    print(f"Doc {i}: TF={tf:.3f}, IDF={idf:.3f}, TF-IDF={tf * idf:.3f}")

Manually computing TF-IDF for the word: 'cat'

Doc 0: TF=0.167, IDF=1.511, TF-IDF=0.252
Doc 1: TF=0.000, IDF=1.511, TF-IDF=0.000
Doc 2: TF=0.000, IDF=1.511, TF-IDF=0.000
Doc 3: TF=0.200, IDF=1.511, TF-IDF=0.302


Compare these values with the `cat` column in the TF-IDF table above — they match the underlying formula sklearn uses, modulo L2 row normalization.

## Summary

| Technique | Captures word order? | Handles common words? | Output |
|---|---|---|---|
| Bag of Words | No | No (all words weighted equally) | Raw counts |
| TF-IDF | No | Yes (downweights common words) | Weighted scores |

Both are simple, interpretable ways to turn text into numeric vectors that can be fed into ML models (e.g., classifiers, clustering algorithms).